## Correlate entry effects

In [1]:
import pandas as pd

In [17]:
entry_23 = pd.read_csv('../results/func_effects/averages/293_2-3_entry_func_effects.csv') 
entry_26 = pd.read_csv('../results/func_effects/averages/293_2-6_entry_func_effects.csv') 
site_map = pd.read_csv('../data/site_numbering_map.csv')
site_map_rename = site_map.rename(columns= {'reference_site': 'site'})

In [18]:
site_map_rename.head()

,sequential_site,site,sequential_wt,region,rbs_region
0,1,11,D,HA1,outside RBS
1,2,12,K,HA1,outside RBS
2,3,13,I,HA1,outside RBS
3,4,14,C,HA1,outside RBS
4,5,15,L,HA1,outside RBS


In [4]:
entry_23.head()

,site,wildtype,mutant,effect,effect_std,times_seen,n_selections
0,100,G,A,0.0496,1.0010,3.0,4
1,100,G,C,-3.9650,0.0270,8.0,4
2,100,G,D,-4.7690,0.0000,20.0,4
3,100,G,E,-4.6740,0.0000,1.0,2
4,100,G,F,-4.6590,0.0985,1.0,4


In [5]:
entry_26.head()

,site,wildtype,mutant,effect,effect_std,times_seen,n_selections
0,100,G,A,-1.079,0.9443,3.0,4
1,100,G,C,-4.524,0.0000,8.0,4
2,100,G,D,-4.925,0.0000,20.0,4
3,100,G,E,-4.844,0.0000,1.0,2
4,100,G,F,-4.859,0.0000,1.0,4


In [6]:
entry_merged = pd.merge(
    entry_23,
    entry_26,
    on=['site', 'wildtype', 'mutant'],
    suffixes=('_23', '_26')
)

entry_merged.head()

,site,wildtype,mutant,effect_23,effect_std_23,times_seen_23,n_selections_23,effect_26,effect_std_26,times_seen_26,n_selections_26
0,100,G,A,0.0496,1.0010,3.0,4,-1.079,0.9443,3.0,4
1,100,G,C,-3.9650,0.0270,8.0,4,-4.524,0.0000,8.0,4
2,100,G,D,-4.7690,0.0000,20.0,4,-4.925,0.0000,20.0,4
3,100,G,E,-4.6740,0.0000,1.0,2,-4.844,0.0000,1.0,2
4,100,G,F,-4.6590,0.0985,1.0,4,-4.859,0.0000,1.0,4


In [21]:
map_entry_merged = pd.merge(
    entry_merged,
    site_map_rename,
    on= ['site']
)
map_entry_merged.head()

,site,wildtype,mutant,effect_23,effect_std_23,times_seen_23,n_selections_23,effect_26,effect_std_26,times_seen_26,n_selections_26,sequential_site,sequential_wt,region,rbs_region
0,100,G,A,0.0496,1.0010,3.0,4,-1.079,0.9443,3.0,4,90,G,HA1,outside RBS
1,100,G,C,-3.9650,0.0270,8.0,4,-4.524,0.0000,8.0,4,90,G,HA1,outside RBS
2,100,G,D,-4.7690,0.0000,20.0,4,-4.925,0.0000,20.0,4,90,G,HA1,outside RBS
3,100,G,E,-4.6740,0.0000,1.0,2,-4.844,0.0000,1.0,2,90,G,HA1,outside RBS
4,100,G,F,-4.6590,0.0985,1.0,4,-4.859,0.0000,1.0,4,90,G,HA1,outside RBS


In [22]:
filtered_entry_merged = map_entry_merged.query(
    'times_seen_23 >= 2 and times_seen_26 >= 2'
).query(
    'effect_std_23 <= 2 and effect_std_26 <= 2'
).query(
    'n_selections_23 >= 2 and n_selections_26 >= 2'
)
filtered_entry_merged.head()

,site,wildtype,mutant,effect_23,effect_std_23,times_seen_23,n_selections_23,effect_26,effect_std_26,times_seen_26,n_selections_26,sequential_site,sequential_wt,region,rbs_region
0,100,G,A,0.0496,1.001,3.00,4,-1.079,0.9443,3.00,4,90,G,HA1,outside RBS
1,100,G,C,-3.9650,0.027,8.00,4,-4.524,0.0000,8.00,4,90,G,HA1,outside RBS
2,100,G,D,-4.7690,0.000,20.00,4,-4.925,0.0000,20.00,4,90,G,HA1,outside RBS
6,100,G,H,-4.7260,0.000,6.25,4,-4.880,0.0000,6.25,4,90,G,HA1,outside RBS
7,100,G,I,-4.3330,0.000,2.50,4,-4.877,0.0000,2.50,4,90,G,HA1,outside RBS


In [40]:
import altair as alt
alt.data_transformers.disable_max_rows()

order=['outside RBS', '130-loop', '150-loop','190-loop','220-loop']

custom_color_scale = alt.Scale(
    domain=['130-loop', '150-loop','190-loop','220-loop', 'outside RBS'],
    range=['#001FFF' ,'#ffa200', '#309E00','#d800b8', '#c8c8c8']  # grey, blue, orange, green
)

alt.Chart(filtered_entry_merged).mark_circle(size=50).encode(
    x = 'effect_23',
    y = 'effect_26',
    tooltip = ['effect_23', 'effect_26', 'site', 'wildtype', 'mutant','rbs_region'],
    color = alt.Color(
        'rbs_region', 
        scale=custom_color_scale,
        sort=order
    )
).properties(width=600, height=600)




alt.Chart(...)

In [54]:
wild = filtered_entry_merged[filtered_entry_merged['rbs_region'] == 'outside RBS']
other = filtered_entry_merged[filtered_entry_merged['rbs_region'] != 'outside RBS']

wild_chart = alt.Chart(wild).mark_circle(size=60).encode(
    x='effect_23',
    y='effect_26',
    tooltip=['effect_23', 'effect_26', 'site', 'wildtype', 'mutant', 'rbs_region'],
    color=alt.value('#C8C8C8')  # Use a fixed color
)

# Chart for other groups
other_chart = alt.Chart(other).mark_circle(size=80).encode(
    x='effect_23',
    y='effect_26',
    tooltip=['effect_23', 'effect_26', 'site', 'wildtype', 'mutant', 'rbs_region'],
    color=alt.Color(
        'rbs_region',
        scale=alt.Scale(
            domain=['130-loop', '150-loop','190-loop','220-loop'],
            range=['#DF8F44' ,'#EFC000', '#B24745','#00A1D5']
        )
    )
)

# Layer them so 'wild' is drawn behind
(wild_chart + other_chart).properties(width=600, height=600)

alt.LayerChart(...)